In [ ]:
# ============================================================
# PLOT DO MAPA DA ESFERA USANDO OS DADOS DO NOVO GERADOR
#
# Este bloco NÃO recalcula energia, picos, FWHM ou Q.
# Ele apenas:
#   1) procura o arquivo final do novo gerador;
#   2) se não encontrar, usa o checkpoint mais recente;
#   3) carrega peaks_data, m_vals e rho_ratio_vals;
#   4) gera o mesmo mapa horizontal do código anterior;
#   5) salva PNG, PDF e SVG e mostra no Colab.
# ============================================================

import os
import glob
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import (
    LinearSegmentedColormap,
    ListedColormap,
    BoundaryNorm,
)
from matplotlib.ticker import FixedLocator, NullLocator, LogFormatterMathtext
from matplotlib.patches import Rectangle


# ============================================================
# 1. GOOGLE DRIVE E LOCAL DOS DADOS
# ============================================================
USE_GOOGLE_DRIVE = True
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_SUBFOLDER = os.path.join(
    "MyDrive",
    "ressonancias_esfera",
    "gerador_Q400_com_zoom",
)
LOCAL_DATA_DIR = os.path.abspath("resultados_gerador_esfera_Q400")

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT, force_remount=False)
        DATA_DIR = os.path.join(DRIVE_MOUNT_POINT, DRIVE_SUBFOLDER)
        print("Google Drive montado com sucesso.")
    except Exception as error:
        print("Não foi possível montar o Google Drive.")
        print("Procurando os dados localmente.")
        print(error)
        DATA_DIR = LOCAL_DATA_DIR
else:
    DATA_DIR = LOCAL_DATA_DIR

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        "A pasta de dados do novo gerador não foi encontrada:\n"
        f"{DATA_DIR}"
    )

FIGURE_DIR = os.path.join(DATA_DIR, "figures")
os.makedirs(FIGURE_DIR, exist_ok=True)


# ============================================================
# 2. LOCALIZA O ARQUIVO FINAL OU O CHECKPOINT
# ============================================================
def newest_file(paths):
    paths = [path for path in paths if os.path.isfile(path)]
    if not paths:
        return None
    return max(paths, key=os.path.getmtime)


final_candidates = glob.glob(
    os.path.join(
        DATA_DIR,
        "data_final_sphere_Q400_xM*_grid*_segment_if_gt*.npz",
    )
)

checkpoint_candidates = glob.glob(
    os.path.join(
        DATA_DIR,
        "checkpoint_sphere_Q400_xM*_grid*_segment_if_gt*.npz",
    )
)

DATA_FILE = newest_file(final_candidates)
DATA_SOURCE = "arquivo final"

if DATA_FILE is None:
    DATA_FILE = newest_file(checkpoint_candidates)
    DATA_SOURCE = "checkpoint"

if DATA_FILE is None:
    raise FileNotFoundError(
        "Nenhum arquivo do novo gerador foi encontrado em:\n"
        f"{DATA_DIR}\n\n"
        "Padrões procurados:\n"
        "  data_final_sphere_Q400_xM*_grid*_segment_if_gt*.npz\n"
        "  checkpoint_sphere_Q400_xM*_grid*_segment_if_gt*.npz"
    )

print(f"Usando {DATA_SOURCE}:")
print(DATA_FILE)


# ============================================================
# 3. CARREGA OS DADOS GERADOS
# ============================================================
with np.load(DATA_FILE, allow_pickle=False) as data:
    required_keys = ["m_vals", "rho_ratio_vals", "done_map"]

    missing_keys = [key for key in required_keys if key not in data]
    if missing_keys:
        raise KeyError(
            "O arquivo não pertence ao novo gerador ou está incompleto.\n"
            "Chaves ausentes: " + ", ".join(missing_keys)
        )

    m_vals = np.asarray(data["m_vals"], dtype=float)
    rho_ratio_vals = np.asarray(data["rho_ratio_vals"], dtype=float)
    done_map = np.asarray(data["done_map"], dtype=bool)

    # O novo gerador salva a contagem final com o nome peaks_data.
    # peak_counts fica como compatibilidade com versões anteriores.
    if "peaks_data" in data:
        peak_counts = np.asarray(data["peaks_data"], dtype=int)
    elif "peak_counts" in data:
        peak_counts = np.asarray(data["peak_counts"], dtype=int)
    else:
        raise KeyError(
            "Não encontrei 'peaks_data' nem 'peak_counts' no arquivo."
        )

    Q_MIN = float(data["Q_MIN"]) if "Q_MIN" in data else 400.0

    if "X_M_MAX" in data:
        X_M_MAX = float(data["X_M_MAX"])
    elif "xM_max" in data:
        X_M_MAX = float(data["xM_max"])
    else:
        X_M_MAX = 1.0

if peak_counts.shape != (len(rho_ratio_vals), len(m_vals)):
    raise ValueError(
        "Dimensão incompatível entre peak_counts e as grades.\n"
        f"peak_counts.shape = {peak_counts.shape}\n"
        f"esperado = {(len(rho_ratio_vals), len(m_vals))}"
    )

if done_map.shape != peak_counts.shape:
    raise ValueError(
        "done_map e peak_counts possuem dimensões diferentes.\n"
        f"done_map.shape = {done_map.shape}\n"
        f"peak_counts.shape = {peak_counts.shape}"
    )

n_done = int(np.count_nonzero(done_map))
n_total = int(done_map.size)
print(f"Curvas concluídas no arquivo: {n_done} / {n_total}")
print(f"Critério carregado: Q >= {Q_MIN:g}")
print(f"Intervalo carregado: x_M <= {X_M_MAX:g}")

if n_done == 0:
    raise RuntimeError("O arquivo ainda não possui nenhuma curva concluída.")

if n_done < n_total:
    print(
        "ATENÇÃO: o arquivo é parcial. "
        "As células ainda não calculadas aparecerão em cinza."
    )


# ============================================================
# 4. BORDAS LOGARÍTMICAS DAS CÉLULAS
# ============================================================
def log_cell_edges(values):
    values = np.asarray(values, dtype=float)

    if values.ndim != 1 or len(values) < 2:
        raise ValueError("A grade precisa ter pelo menos dois valores.")

    if np.any(~np.isfinite(values)) or np.any(values <= 0.0):
        raise ValueError("A grade logarítmica deve ser positiva e finita.")

    log_values = np.log10(values)
    edges = np.empty(len(values) + 1, dtype=float)

    edges[1:-1] = 0.5 * (log_values[:-1] + log_values[1:])
    edges[0] = log_values[0] - 0.5 * (log_values[1] - log_values[0])
    edges[-1] = log_values[-1] + 0.5 * (
        log_values[-1] - log_values[-2]
    )

    return 10.0**edges


def make_5_cbar_ticks(vmin, vmax):
    """Retorna até cinco marcas inteiras igualmente espaçadas."""
    vmin = int(vmin)
    vmax = int(vmax)

    if vmax <= vmin:
        return np.array([vmin], dtype=int)

    ticks = np.rint(np.linspace(vmin, vmax, 5)).astype(int)
    ticks[0] = vmin
    ticks[-1] = vmax
    return np.unique(ticks)


# ============================================================
# 5. ESTILO VISUAL DO PLOT ORIGINAL
# ============================================================
BOX_COLOR = "#600B00"
FIGSIZE_SPHERE_MAP = (15.0, 5.5)
LABEL_FONTSIZE = 23
TICK_FONTSIZE = 15
CBAR_LABEL_FONTSIZE = 19
CBAR_TICK_FONTSIZE = 15
BOX_LINEWIDTH = 1.8
CBAR_BOX_LINEWIDTH = 1.8
CELL_LINECOLOR = "white"
CELL_LINEWIDTH = 0.42
FIGURE_DPI = 600

# A contagem usa escala LINEAR, como no mapa original.
# O valor zero recebe uma faixa exclusivamente branca; as contagens
# positivas usam a paleta cinza-azul-preto sem transformação logarítmica.

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
})

positive_cmap = LinearSegmentedColormap.from_list(
    "sphere_resonance_positive_cmap",
    [
        (0.00, "#EFEFEF"),
        (0.08, "#E3E5E8"),
        (0.18, "#CED3DA"),
        (0.30, "#AEB8C8"),
        (0.48, "#6E88B3"),
        (0.65, "#3E659A"),
        (0.82, "#1C3D57"),
        (0.94, "#08151C"),
        (1.00, "#000000"),
    ],
    N=256,
)

# A primeira cor é reservada SOMENTE para N=0.
# As outras 256 cores representam linearmente as contagens positivas.
zero_white_colors = np.vstack([
    np.array([[1.0, 1.0, 1.0, 1.0]]),
    positive_cmap(np.linspace(0.0, 1.0, 256)),
])
sphere_resonance_cmap = ListedColormap(
    zero_white_colors,
    name="sphere_resonance_zero_white",
)

# Células ainda não calculadas continuam cinza em uma camada separada.
incomplete_cmap = ListedColormap(["#D8D8D8"])
incomplete_norm = BoundaryNorm([-0.5, 0.5], incomplete_cmap.N)


# ============================================================
# 6. PREPARAÇÃO DO MAPA
# ============================================================
m_edges = log_cell_edges(m_vals)
rho_edges = log_cell_edges(rho_ratio_vals)

# Camada das células ainda não calculadas.
# As células concluídas ficam mascaradas nessa camada.
incomplete_map = np.ma.masked_all(peak_counts.shape, dtype=float)
incomplete_map[~done_map] = 0.0

# O mapa principal inclui o zero. Apenas as células não calculadas são mascaradas.
plot_counts = np.ma.array(
    peak_counts.astype(float),
    mask=~done_map,
)

completed_counts = peak_counts[done_map]
max_peak_count = int(np.max(completed_counts))
print("Normalização de cores: linear, com zero exclusivamente branco.")
print(f"Faixa da colorbar: 0 até {max_peak_count}.")

# Primeiro intervalo: [-0.5, 0.5] -> branco, exclusivamente para N=0.
# Os 256 intervalos seguintes distribuem linearmente N>0 até o máximo.
positive_boundaries = np.linspace(
    0.5,
    max(1.5, max_peak_count + 0.5),
    257,
)
color_boundaries = np.concatenate(([-0.5], positive_boundaries))
norm = BoundaryNorm(
    color_boundaries,
    ncolors=sphere_resonance_cmap.N,
    clip=False,
)

x_ticks = [1.0e-1, 1.0e0, 1.0e1, 1.0e2, 1.0e3]
y_ticks = [1.0e-4, 1.0e-2, 1.0e0, 1.0e2, 1.0e3]
cbar_ticks = make_5_cbar_ticks(0, max_peak_count)


# ============================================================
# 7. GRÁFICO
# ============================================================
fig, ax = plt.subplots(figsize=FIGSIZE_SPHERE_MAP)

# Células ainda não calculadas em cinza.
ax.pcolormesh(
    m_edges,
    rho_edges,
    incomplete_map,
    cmap=incomplete_cmap,
    norm=incomplete_norm,
    shading="flat",
    edgecolors=CELL_LINECOLOR,
    linewidth=CELL_LINEWIDTH,
    antialiased=True,
)

# Mapa concluído: zero branco e contagens positivas em escala linear.
mesh = ax.pcolormesh(
    m_edges,
    rho_edges,
    plot_counts,
    cmap=sphere_resonance_cmap,
    norm=norm,
    shading="flat",
    edgecolors=CELL_LINECOLOR,
    linewidth=CELL_LINEWIDTH,
    antialiased=True,
)


# ============================================================
# 8. RETÂNGULOS DE REFERÊNCIA
# ============================================================
air_rect = Rectangle(
    (2.0, 1.0e-3),
    4.0,
    1.0e-3,
    linewidth=2.5,
    edgecolor="#D62728",
    facecolor="none",
    linestyle="--",
)
ax.add_patch(air_rect)

ax.text(
    6.1,
    2.1e-3,
    "Ar",
    fontsize=16,
    color="#D62728",
    ha="left",
    va="bottom",
    weight="bold",
)

oil_rect = Rectangle(
    (0.9, 0.7),
    0.5,
    0.4,
    linewidth=2.5,
    edgecolor="#1F77B4",
    facecolor="none",
    linestyle="--",
)
ax.add_patch(oil_rect)

ax.text(
    1.45,
    1.15,
    "Óleo",
    fontsize=16,
    color="#1F77B4",
    ha="left",
    va="bottom",
    weight="bold",
)


# ============================================================
# 9. EIXOS E TICKS
# ============================================================
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(m_vals[0], m_vals[-1])
ax.set_ylim(rho_ratio_vals[0], rho_ratio_vals[-1])

ax.set_xlabel(
    r"Razão de velocidades $(m=c_M/c_S)$",
    fontsize=LABEL_FONTSIZE,
)
ax.set_ylabel(
    r"Razão de densidades $(\rho_S/\rho_M)$",
    fontsize=LABEL_FONTSIZE,
)

ax.xaxis.set_major_locator(FixedLocator(x_ticks))
ax.yaxis.set_major_locator(FixedLocator(y_ticks))
ax.xaxis.set_major_formatter(LogFormatterMathtext())
ax.yaxis.set_major_formatter(LogFormatterMathtext())
ax.xaxis.set_minor_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

ax.tick_params(
    axis="both",
    which="major",
    labelsize=TICK_FONTSIZE,
    width=BOX_LINEWIDTH,
    length=7,
    color=BOX_COLOR,
    labelcolor="black",
)

for spine in ax.spines.values():
    spine.set_color(BOX_COLOR)
    spine.set_linewidth(BOX_LINEWIDTH)


# ============================================================
# 10. COLORBAR
# ============================================================
cbar = fig.colorbar(
    mesh,
    ax=ax,
    ticks=cbar_ticks,
    pad=0.025,
    fraction=0.032,
    boundaries=color_boundaries,
    spacing="proportional",
)

cbar.set_label(
    r"Número de ressonâncias $(x_M \in [0,1])$",
    fontsize=CBAR_LABEL_FONTSIZE,
    labelpad=30,
    rotation=-90,
)

cbar.ax.tick_params(
    labelsize=CBAR_TICK_FONTSIZE,
    width=CBAR_BOX_LINEWIDTH,
    length=5,
    color=BOX_COLOR,
    labelcolor="black",
)
cbar.minorticks_off()
cbar.outline.set_edgecolor(BOX_COLOR)
cbar.outline.set_linewidth(CBAR_BOX_LINEWIDTH)


# ============================================================
# 11. SALVAR E MOSTRAR NO COLAB
# ============================================================
plt.tight_layout()

output_stem = os.path.join(
    FIGURE_DIR,
    "mapa_ressonancias_xM_0_1_linear_zero_branco",
)

fig.savefig(
    output_stem + ".png",
    dpi=FIGURE_DPI,
    bbox_inches="tight",
    pad_inches=0.05,
)
fig.savefig(
    output_stem + ".pdf",
    bbox_inches="tight",
    pad_inches=0.05,
)
fig.savefig(
    output_stem + ".svg",
    bbox_inches="tight",
    pad_inches=0.05,
)

plt.show()
plt.close(fig)

print("\n" + "=" * 80)
print("PLOT FINALIZADO SEM RECALCULAR OS ESPECTROS")
print("=" * 80)
print(f"Arquivo de dados: {DATA_FILE}")
print(f"Máximo de ressonâncias encontrado: {max_peak_count}")
print(f"Curvas concluídas: {n_done} / {n_total}")
print("Figuras salvas em:")
print(output_stem + ".png")
print(output_stem + ".pdf")
print(output_stem + ".svg")
